# TRI^P Baseline Builder (general_enterprise_nids)

This notebook builds a **context-aware baseline** for the TRI^P metric (Volume, Temporal, Protocol mix)
using your **MAWI**, **UGR'16**, and **UNIBS** flow sources.

It is Windows-friendly: just edit the path globs in the **Config** cell using either raw strings (r"C:\\path\\*")
or forward slashes ("C:/path/*").

### What it does
1. Loads Zeek `conn.log` files (JSON-lines or ASCII with `#fields`) from MAWI & UNIBS
2. Loads UGR'16 weekly CSV (the `*.uniqblacklistremoved` file)
3. Normalizes to a common flow schema: `bytes, pkts, duration, start_ts, proto, service`
4. Computes baseline histograms/CDFs + temporal stats and protocol/service mixes
5. Saves a YAML baseline in a `baselines/` folder

**Dependencies:** `pandas`, `numpy`, `pyyaml`, `python-dateutil`


In [1]:
# If needed, uncomment to install dependencies in your Jupyter environment
# !pip install pandas numpy pyyaml python-dateutil
import os, glob, gzip, math
from datetime import datetime
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import yaml
from dateutil import parser as dtparser


## Config (edit me)

- **Windows users:** use either raw strings (e.g., `r"C:\\data\\mawi\\conn\\*\\conn.log.gz"`) or forward slashes (e.g., `"C:/data/mawi/conn/*/conn.log.gz"`).
- You can set `UGR16_USE_DAYS` to a list like `["2016-05-01", "2016-05-02"]` to restrict to specific days.
- You can set `UGR16_SAMPLE_FRACTION = 0.10` to sample 10% for memory savings.


In [13]:
# Paths prefilled from your message (Linux-style). Change as needed for Windows.
MAWI_GLOB  = "E:/data/mawi/conn/*/conn.log.gz"
UGR16_FILE = "E:/data/ugr16/may_week1_csv/uniq/may.week1.csv.uniqblacklistremoved"
UNIBS_GLOB = "E:/data/unibs/comm/*/conn.log.gz"  # change to your real UNIBS conn path

PROFILE_NAME   = "general_enterprise_nids"
OUTPUT_DIR     = "baselines"
BINS_LOG10     = 50        # number of bins for log10 histograms
BURST_WINDOW_S = 10        # window size (s) for Fano factor
UGR16_USE_DAYS: Optional[List[str]] = [2016-05-01]  # e.g., ["2016-05-01","2016-05-02"]; None = use full week
UGR16_SAMPLE_FRACTION: Optional[float] = 0.10  # e.g., 0.10 for 10%; None = use all


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3019487385.py, line 10)

## Helpers: reading Zeek `conn.log` (JSON-lines or ASCII with `#fields`)

In [5]:
def _open_auto(path: str):
    if path.endswith('.gz'):
        return gzip.open(path, 'rb')
    return open(path, 'rb')

def _peek_first_non_ws_char(path: str):
    with _open_auto(path) as f:
        first = f.read(4096)
    try:
        s = first.decode('utf-8', errors='ignore')
    except Exception:
        return None
    for ch in s:
        if not ch.isspace():
            return ch
    return None

def read_zeek_conn_any(path: str) -> pd.DataFrame:
    """Read a Zeek conn.log (JSON lines or ASCII with #fields) into a DataFrame."""
    first = _peek_first_non_ws_char(path)
    if first == '{':
        # JSON lines
        with _open_auto(path) as f:
            return pd.read_json(f, lines=True)
    # ASCII TSV with header lines starting with '#'
    fields = None
    with _open_auto(path) as f:
        for raw in f:
            line = raw.decode('utf-8', errors='ignore').rstrip('\n')
            if line.startswith('#fields'):
                parts = line.split('\t')
                fields = parts[1:]
                break
    if not fields:
        # Fallback: try reading as tab-separated without header
        with _open_auto(path) as f:
            return pd.read_csv(f, sep='\t', header=None, engine='python', comment='#')
    with _open_auto(path) as f:
        df = pd.read_csv(f, sep='\t', header=None, names=fields, engine='python', comment='#')
    return df

def normalize_zeek_conn(df: pd.DataFrame) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}
    # bytes
    if 'orig_bytes' in cols and 'resp_bytes' in cols:
        b = pd.to_numeric(df[cols['orig_bytes']], errors='coerce').fillna(0) + \
            pd.to_numeric(df[cols['resp_bytes']], errors='coerce').fillna(0)
    elif 'bytes' in cols:
        b = pd.to_numeric(df[cols['bytes']], errors='coerce')
    else:
        b = np.nan
    # packets
    if 'orig_pkts' in cols and 'resp_pkts' in cols:
        p = pd.to_numeric(df[cols['orig_pkts']], errors='coerce').fillna(0) + \
            pd.to_numeric(df[cols['resp_pkts']], errors='coerce').fillna(0)
    elif 'pkts' in cols:
        p = pd.to_numeric(df[cols['pkts']], errors='coerce')
    else:
        p = np.nan
    # duration
    d = pd.to_numeric(df[cols.get('duration','duration')], errors='coerce') if 'duration' in cols else pd.Series(np.nan, index=df.index)
    # timestamp (start)
    if 'ts' in cols:
        ts = pd.to_numeric(df[cols['ts']], errors='coerce')  # Zeek ts = epoch seconds (float)
    elif 'starttime' in cols:
        ts = pd.to_datetime(df[cols['starttime']], errors='coerce').astype('int64')/1e9
    else:
        ts = np.nan
    # l4 protocol
    proto = df[cols['proto']].astype(str).str.lower() if 'proto' in cols else 'unknown'
    # service/app (optional)
    service = df[cols['service']].astype(str).str.lower() if 'service' in cols else None
    out = pd.DataFrame({
        'bytes': b,
        'pkts':  p,
        'duration': d,
        'start_ts': ts,
        'proto': proto,
        'service': service,
    })
    return out.replace([np.inf, -np.inf], np.nan)


## UGR'16 reader/normalizer

In [7]:
def read_ugr16_csv(path: str, use_days: Optional[List[str]] = None, sample_frac: Optional[float] = None) -> pd.DataFrame:
    """Read UGR'16 preprocessed CSV (uniqblacklistremoved). Canonical order:
    te, td, sa, da, sp, dp, pr, flg, fwd, stos, pkt, byt [, label]
    """
    if sample_frac is not None and (0 < sample_frac < 1):
        iter_csv = pd.read_csv(path, header=None, chunksize=1_000_000, low_memory=False)
        blocks = []
        rng = np.random.default_rng(42)
        for chunk in iter_csv:
            if use_days:
                te_col = chunk.iloc[:,0].astype(str)
                mask = None
                for day in use_days:
                    mask = te_col.str.startswith(day) if mask is None else (mask | te_col.str.startswith(day))
                chunk = chunk[mask]
            keep = rng.random(len(chunk)) <= sample_frac
            if keep.any():
                blocks.append(chunk[keep])
        df = pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame()
    else:
        df = pd.read_csv(path, header=None, low_memory=False)
        if use_days:
            te_col = df.iloc[:,0].astype(str)
            mask = None
            for day in use_days:
                mask = te_col.str.startswith(day) if mask is None else (mask | te_col.str.startswith(day))
            df = df[mask]
    if df.empty:
        return pd.DataFrame(columns=['bytes','pkts','duration','start_ts','proto'])
    ncols = df.shape[1]
    base_cols = ["te","td","sa","da","sp","dp","pr","flg","fwd","stos","pkt","byt"]
    cols = base_cols[:min(ncols, len(base_cols))] + (["label"] if ncols > len(base_cols) else [])
    df.columns = cols
    te = pd.to_datetime(df["te"], errors='coerce').astype('int64')/1e9
    td = pd.to_numeric(df["td"], errors='coerce')
    flows = pd.DataFrame({
        'bytes': pd.to_numeric(df['byt'], errors='coerce'),
        'pkts':  pd.to_numeric(df['pkt'], errors='coerce'),
        'duration': td,
        'start_ts': te - td.fillna(0),
        'proto': df['pr'].astype(str).str.lower(),
        'service': None,
    })
    return flows.replace([np.inf, -np.inf], np.nan)


## Metrics & utilities

In [9]:
def _winsorize(series: pd.Series, q_low=0.005, q_high=0.995) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce').dropna().astype(float)
    if s.empty: return s
    lo, hi = s.quantile([q_low, q_high])
    return s.clip(lower=lo, upper=hi)

def _log10_hist_cdf(series: pd.Series, n_bins=50):
    s = _winsorize(series)
    s = s[s > 0]
    if s.empty: return [], []
    logv = np.log10(s.values)
    hist, edges = np.histogram(logv, bins=n_bins, density=True)
    cdf = np.cumsum(hist) / np.sum(hist)
    return edges.tolist(), cdf.tolist()

def compute_volume(flows: pd.DataFrame, n_bins: int) -> Dict[str, Dict[str, List[float]]]:
    out = {}
    for col, key in [("bytes","bytes_per_flow"), ("pkts","pkts_per_flow"), ("duration","flow_duration")]:
        edges, cdf = _log10_hist_cdf(flows[col], n_bins)
        out[key] = {"bins_log10": edges, "cdf": cdf}
    return out

def compute_iat(flows: pd.DataFrame, n_bins: int):
    ts = pd.to_numeric(flows['start_ts'], errors='coerce').dropna()
    if ts.empty: return None
    ts = ts.sort_values()
    iat = ts.diff().dropna()
    iat = iat[iat > 0]
    if iat.empty: return None
    edges, cdf = _log10_hist_cdf(iat, n_bins)
    return {"bins_log10": edges, "cdf": cdf}

def compute_burstiness(flows: pd.DataFrame, window_s: int = 10):
    ts = pd.to_numeric(flows['start_ts'], errors='coerce').dropna()
    if ts.empty: return None
    t0 = ts.min()
    buckets = np.floor((ts - t0) / window_s).astype(int)
    counts = pd.Series(1, index=buckets).groupby(level=0).sum().astype(float)
    lam = counts.mean(); var = counts.var(ddof=0)
    if lam <= 0: return None
    return {"window_s": int(window_s), "fano_ref": float(var/lam)}

def compute_diurnal(flows: pd.DataFrame):
    ts = pd.to_numeric(flows['start_ts'], errors='coerce').dropna()
    if ts.empty: return None
    dt = pd.to_datetime(ts, unit='s')
    hours = dt.dt.hour
    counts = hours.value_counts().reindex(range(24), fill_value=0).astype(float)
    total = counts.sum()
    if total <= 0: return None
    return (counts / total).tolist()

def _normalize_proto(val: str) -> str:
    v = str(val).strip().lower()
    mapping_num = {'6':'tcp','17':'udp','1':'icmp'}
    if v in mapping_num: return mapping_num[v]
    if 'tcp' in v: return 'tcp'
    if 'udp' in v: return 'udp'
    if 'icmp' in v: return 'icmp'
    return v or 'unknown'

def proto_mix(flows: pd.DataFrame) -> Dict[str,float]:
    if 'proto' not in flows.columns:
        return {'unknown':1.0}
    norm = flows['proto'].astype(str).map(_normalize_proto)
    counts = norm.value_counts()
    probs = (counts / counts.sum()).to_dict()
    return {str(k): float(v) for k,v in probs.items()}

def service_mix(flows: pd.DataFrame, top_k: int = 15):
    if 'service' not in flows.columns: return None
    s = flows['service'].dropna().astype(str).str.lower()
    s = s[(s != '-') & (s != 'none')]
    if s.empty: return None
    counts = s.value_counts()
    if len(counts) > top_k:
        top = counts.head(top_k)
        other = counts.iloc[top_k:].sum()
        counts = pd.concat([top, pd.Series({'other': other})])
    probs = (counts / counts.sum()).to_dict()
    return {str(k): float(v) for k,v in probs.items()}


## Load sources (MAWI, UNIBS, UGR'16)

In [15]:
def load_all_sources(mawi_glob: str, unibs_glob: str, ugr16_file: str,
                     ugr16_days=None, ugr16_sample_frac=None) -> pd.DataFrame:
    frames = []
    # MAWI
    for path in glob.glob(mawi_glob):
        try:
            df_raw = read_zeek_conn_any(path)
            df = normalize_zeek_conn(df_raw)
            frames.append(df)
        except Exception as e:
            print(f"[WARN] Could not read MAWI file {path}: {e}")
    # UNIBS
    for path in glob.glob(unibs_glob):
        try:
            df_raw = read_zeek_conn_any(path)
            df = normalize_zeek_conn(df_raw)
            frames.append(df)
        except Exception as e:
            print(f"[WARN] Could not read UNIBS file {path}: {e}")
    # UGR'16
    if os.path.exists(ugr16_file):
        try:
            df_ugr = read_ugr16_csv(ugr16_file, use_days=ugr16_days, sample_frac=ugr16_sample_frac)
            frames.append(df_ugr)
        except Exception as e:
            print(f"[WARN] Could not read UGR'16 file {ugr16_file}: {e}")
    if not frames:
        raise FileNotFoundError("No input flows found. Check your globs and UGR'16 path.")
    flows = pd.concat(frames, ignore_index=True)
    flows = flows.replace([np.inf, -np.inf], np.nan)
    flows
    return flows

flows = load_all_sources(MAWI_GLOB, UNIBS_GLOB, UGR16_FILE, UGR16_USE_DAYS, UGR16_SAMPLE_FRACTION)
print("Loaded flows:", len(flows))
flows.head()

[WARN] Could not read UGR'16 file E:/data/ugr16/may_week1_csv/uniq/may.week1.csv.uniqblacklistremoved: Error tokenizing data. C error: out of memory
Loaded flows: 10296350


,bytes,pkts,duration,start_ts,proto,service
0,0.0,1,NaN,1.754024e+09,tcp,nan
1,0.0,1,NaN,1.754024e+09,tcp,nan
2,0.0,1,NaN,1.754024e+09,tcp,nan
3,0.0,1,NaN,1.754024e+09,tcp,nan
4,0.0,1,NaN,1.754024e+09,tcp,nan


## Build baseline dict

In [17]:
def build_baseline(flows: pd.DataFrame) -> Dict:
    coverage_hours = None
    if flows['start_ts'].notna().any():
        ts = pd.to_numeric(flows['start_ts'], errors='coerce').dropna()
        if not ts.empty:
            coverage_hours = float((ts.max() - ts.min()) / 3600.0)
    volume = compute_volume(flows, n_bins=BINS_LOG10)
    iat = compute_iat(flows, n_bins=BINS_LOG10)
    burst = compute_burstiness(flows, window_s=BURST_WINDOW_S)
    diurnal = compute_diurnal(flows)
    p_mix = proto_mix(flows)
    s_mix = service_mix(flows)
    now = datetime.utcnow()
    baseline = {
        'profile': PROFILE_NAME,
        'created_at_utc': now.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'version': now.strftime('%Y-%m'),
        'sources': {
            'mawi_glob': MAWI_GLOB,
            'ugr16_file': UGR16_FILE,
            'unibs_glob': UNIBS_GLOB,
        },
        'stats': {
            'n_flows': int(len(flows)),
            'coverage_hours': coverage_hours,
        },
        'metrics': {
            **volume,
            'iat_flows': iat,
            'burstiness': burst,
            'diurnal_shape_hourly': diurnal,
            'protocol_mix': p_mix,
            'service_mix': s_mix,
        },
        'notes': {
            'winsorize_quantiles': [0.005, 0.995],
            'bins': BINS_LOG10,
            'burst_window_s': BURST_WINDOW_S,
            'omitted_temporal': bool(iat is None or diurnal is None or burst is None),
            'ugr16_sample_fraction': UGR16_SAMPLE_FRACTION,
            'ugr16_days_used': UGR16_USE_DAYS,
        },
    }
    return baseline

baseline = build_baseline(flows)
baseline_keys = list(baseline['metrics'].keys())
print("Metrics available:", baseline_keys)
print("n_flows:", baseline['stats']['n_flows'])
print("coverage_hours:", baseline['stats']['coverage_hours'])

Metrics available: ['bytes_per_flow', 'pkts_per_flow', 'flow_duration', 'iat_flows', 'burstiness', 'diurnal_shape_hourly', 'protocol_mix', 'service_mix']
n_flows: 10296350
coverage_hours: 48.050283657775985


## Save YAML baseline

In [19]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, f"{PROFILE_NAME}_baseline_{datetime.utcnow().strftime('%Y%m%d')}.yaml")
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(baseline, f, sort_keys=False, allow_unicode=True)
out_path

'baselines\\general_enterprise_nids_baseline_20250902.yaml'

### Done
The path above is your baseline YAML file. You can now plug it into your TRI^P scorer.

In [12]:
# If needed, uncomment to install dependencies in your Jupyter environment
# !pip install pandas numpy pyyaml python-dateutil
import os, glob, gzip
from datetime import datetime
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import yaml

# ---------- CONFIG ----------
MAWI_GLOB  = r"E:/data/mawi/conn/*/conn.log.gz"
UGR16_FILE = r"E:/data/ugr16/may_week1_csv/uniq/may.week1.csv.uniqblacklistremoved"
UNIBS_GLOB = r"E:/data/unibs/comm/*/conn.log.gz"

PROFILE_NAME   = "general_enterprise_nids"
OUTPUT_DIR     = "baselines"

BINS_LOG10     = 50         # bins for log10 histograms
BURST_WINDOW_S = 10         # seconds for Fano
# IMPORTANT: use strings like "2016-05-01", or None to use full file
UGR16_USE_DAYS: Optional[List[str]] = None  # e.g., ["2016-05-01","2016-05-02"]
UGR16_SAMPLE_FRACTION: Optional[float] = 0.10  # set None to use all
RNG_SEED = 42

# ---------- helpers ----------
def _open_auto(path: str):
    return gzip.open(path, "rb") if path.endswith(".gz") else open(path, "rb")

def _peek_first_non_ws_char(path: str):
    with _open_auto(path) as f:
        first = f.read(4096)
    try:
        s = first.decode("utf-8", errors="ignore")
    except Exception:
        return None
    for ch in s:
        if not ch.isspace():
            return ch
    return None

def _ci(df) -> Dict[str,str]:
    return {str(c).lower(): c for c in df.columns}

def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.replace(",","").str.strip(), errors="coerce")

def _winsorize(series: pd.Series, q_low=0.005, q_high=0.995) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").dropna().astype(float)
    if s.empty: return s
    lo, hi = s.quantile([q_low, q_high])
    return s.clip(lower=lo, upper=hi)

def _log10_hist_cdf(series: pd.Series, n_bins=50):
    s = _winsorize(series)
    s = s[s > 0]
    if s.empty: return [], []
    logv = np.log10(s.to_numpy())
    hist, edges = np.histogram(logv, bins=n_bins, density=True)
    if np.sum(hist) == 0:
        return edges.tolist(), [0.0]*(len(edges)-1)
    cdf = np.cumsum(hist) / np.sum(hist)
    return edges.tolist(), cdf.tolist()

# --------- service inference from port (when Zeek 'service' is '-') ----------
_PORT2SVC = {
    53:"dns", 80:"http", 443:"https", 123:"ntp", 20:"ftp-data", 21:"ftp", 22:"ssh",
    23:"telnet", 25:"smtp", 110:"pop3", 143:"imap", 69:"tftp", 161:"snmp",
    995:"pop3s", 993:"imaps", 587:"submission", 8080:"http-alt", 3306:"mysql",
    3389:"rdp", 5060:"sip", 5353:"mdns", 1900:"ssdp", 554:"rtsp", 8883:"mqtts", 1883:"mqtt"
}
def _infer_service_from_port(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    return s.map(lambda x: _PORT2SVC.get(int(x), None) if pd.notna(x) else None)

def _normalize_proto(v: str) -> str:
    v = str(v).strip().lower()
    if v in ("6","tcp"): return "tcp"
    if v in ("17","udp"): return "udp"
    if v in ("1","icmp"): return "icmp"
    return v or "unknown"

# ---------- readers / normalizers ----------
def read_zeek_conn_any(path: str) -> pd.DataFrame:
    """Read Zeek conn.log (JSON lines or ASCII TSV with #fields)."""
    first = _peek_first_non_ws_char(path)
    if first == "{":
        with _open_auto(path) as f:
            return pd.read_json(f, lines=True)
    # ASCII TSV with Zeek headers
    fields = None
    with _open_auto(path) as f:
        for raw in f:
            line = raw.decode("utf-8", errors="ignore").rstrip("\n")
            if line.startswith("#fields"):
                parts = line.split("\t")
                fields = parts[1:]
                break
    if not fields:
        with _open_auto(path) as f:
            return pd.read_csv(f, sep="\t", header=None, engine="python", comment="#")
    with _open_auto(path) as f:
        df = pd.read_csv(f, sep="\t", header=None, names=fields, engine="python", comment="#")
    return df

def normalize_zeek_conn(df: pd.DataFrame) -> pd.DataFrame:
    cols = _ci(df)

    # bytes
    if "orig_bytes" in cols and "resp_bytes" in cols:
        b = _to_num(df[cols["orig_bytes"]]).fillna(0) + _to_num(df[cols["resp_bytes"]]).fillna(0)
    elif "bytes" in cols:
        b = _to_num(df[cols["bytes"]])
    else:
        b = pd.Series(np.nan, index=df.index)

    # packets
    if "orig_pkts" in cols and "resp_pkts" in cols:
        p = _to_num(df[cols["orig_pkts"]]).fillna(0) + _to_num(df[cols["resp_pkts"]]).fillna(0)
    elif "pkts" in cols:
        p = _to_num(df[cols["pkts"]])
    else:
        p = pd.Series(np.nan, index=df.index)

    # duration (seconds, float)
    d = _to_num(df[cols["duration"]]) if "duration" in cols else pd.Series(np.nan, index=df.index)

    # start timestamp (epoch seconds float)
    if "ts" in cols:
        ts = _to_num(df[cols["ts"]])  # Zeek 'ts' is epoch sec
    elif "starttime" in cols:
        ts = pd.to_datetime(df[cols["starttime"]], errors="coerce", utc=True).view("int64")/1e9
    else:
        ts = pd.Series(np.nan, index=df.index)

    # protocol
    proto = df[cols["proto"]].astype(str).map(_normalize_proto) if "proto" in cols else pd.Series("unknown", index=df.index)

    # service (prefer Zeek 'service'; else infer from responder port)
    if "service" in cols:
        service = df[cols["service"]].astype(str).str.lower()
        # If service is "-" or "none", try to infer from resp_p/id.resp_p
        if ("resp_p" in cols) or ("id.resp_p" in cols):
            resp_p_col = cols["resp_p"] if "resp_p" in cols else cols["id.resp_p"]
            inferred = _infer_service_from_port(df[resp_p_col])
            service = service.where(~service.isin(["-","none",""]), inferred)
    else:
        service = None
        if ("resp_p" in cols) or ("id.resp_p" in cols):
            resp_p_col = cols["resp_p"] if "resp_p" in cols else cols["id.resp_p"]
            service = _infer_service_from_port(df[resp_p_col])

    out = pd.DataFrame({
        "bytes": b,
        "pkts": p,
        "duration": d,
        "start_ts": ts,
        "proto": proto,
        "service": service if isinstance(service, pd.Series) else None,
    })
    return out.replace([np.inf, -np.inf], np.nan)

def read_ugr16_csv(path: str, use_days: Optional[List[str]] = None, sample_frac: Optional[float] = None) -> pd.DataFrame:
    """
    Read UGR'16 preprocessed CSV (uniqblacklistremoved).
    Canonical columns: te, td, sa, da, sp, dp, pr, flg, fwd, stos, pkt, byt [, label]
    """
    rng = np.random.default_rng(RNG_SEED)
    def _maybe_filter_days(df):
        if not use_days: return df
        te_col = df.iloc[:,0].astype(str)
        mask = False
        for day in use_days:
            day = str(day)  # ensure string
            mask = (te_col.str.startswith(day)) | (mask if isinstance(mask, pd.Series) else False)
        return df[mask]

    if sample_frac is not None and (0 < sample_frac < 1):
        blocks = []
        for chunk in pd.read_csv(path, header=None, chunksize=1_000_000, low_memory=False):
            chunk = _maybe_filter_days(chunk)
            if len(chunk)==0: continue
            keep = rng.random(len(chunk)) <= sample_frac
            if keep.any(): blocks.append(chunk[keep])
        df = pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame()
    else:
        df = pd.read_csv(path, header=None, low_memory=False)
        df = _maybe_filter_days(df)

    if df.empty:
        return pd.DataFrame(columns=["bytes","pkts","duration","start_ts","proto","service"])

    ncols = df.shape[1]
    base_cols = ["te","td","sa","da","sp","dp","pr","flg","fwd","stos","pkt","byt"]
    cols = base_cols[:min(ncols, len(base_cols))] + (["label"] if ncols > len(base_cols) else [])
    df.columns = cols

    te = pd.to_datetime(df["te"], errors="coerce", utc=True).view("int64")/1e9
    td = _to_num(df["td"])  # seconds
    flows = pd.DataFrame({
        "bytes": _to_num(df["byt"]),
        "pkts":  _to_num(df["pkt"]),
        "duration": td,
        "start_ts": te - td.fillna(0),
        "proto": df["pr"].astype(str).map(_normalize_proto),
        "service": None,  # no robust service info in UGR'16
    })
    return flows.replace([np.inf, -np.inf], np.nan)

# ---------- metrics ----------
def compute_volume(flows: pd.DataFrame, n_bins: int) -> Dict[str, Dict[str, list]]:
    out = {}
    for col, key in [("bytes","bytes_per_flow"), ("pkts","pkts_per_flow"), ("duration","flow_duration")]:
        edges, cdf = _log10_hist_cdf(flows[col], n_bins)
        if edges:  # only include if non-empty
            out[key] = {"bins_log10": edges, "cdf": cdf}
    return out

def compute_iat(flows: pd.DataFrame, n_bins: int):
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty: return None
    ts = ts.sort_values()
    iat = ts.diff().dropna()
    iat = iat[iat > 0]
    if iat.empty: return None
    edges, cdf = _log10_hist_cdf(iat, n_bins)
    if not edges: return None
    return {"bins_log10": edges, "cdf": cdf}

def compute_burstiness(flows: pd.DataFrame, window_s: int = 10):
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty: return None
    t0 = ts.min()
    buckets = np.floor((ts - t0) / window_s).astype(int)
    counts = pd.Series(1, index=buckets).groupby(level=0).sum().astype(float)
    lam = counts.mean(); var = counts.var(ddof=0)
    if lam <= 0: return None
    return {"window_s": int(window_s), "fano_ref": float(var/lam)}

def compute_diurnal(flows: pd.DataFrame):
    ts = _to_num(flows["start_ts"]).dropna()
    if ts.empty: return None
    hours = pd.to_datetime(ts, unit="s", utc=True).dt.hour
    counts = hours.value_counts().reindex(range(24), fill_value=0).astype(float)
    total = counts.sum()
    if total <= 0: return None
    return (counts / total).tolist()

def proto_mix(flows: pd.DataFrame) -> Dict[str,float]:
    if "proto" not in flows.columns: return {"unknown": 1.0}
    norm = flows["proto"].astype(str).map(_normalize_proto)
    counts = norm.value_counts()
    if counts.sum() == 0: return {"unknown": 1.0}
    probs = (counts / counts.sum()).to_dict()
    return {str(k): float(v) for k,v in probs.items()}

def service_mix(flows: pd.DataFrame, top_k: int = 15):
    if "service" not in flows.columns: return None
    s = flows["service"].dropna().astype(str).str.lower()
    s = s[(s != "-") & (s != "none") & (s != "")]
    if s.empty: return None
    counts = s.value_counts()
    if len(counts) > top_k:
        top = counts.head(top_k)
        other = counts.iloc[top_k:].sum()
        counts = pd.concat([top, pd.Series({"other": other})])
    probs = (counts / counts.sum()).to_dict()
    return {str(k): float(v) for k,v in probs.items()}

# Add this helper near your other metric helpers
def compute_flow_iat_mean_proxy(flows: pd.DataFrame, n_bins=50):
    if not {"duration","pkts"}.issubset(flows.columns):
        return None
    dur = _to_num(flows["duration"]).clip(lower=0)
    pk  = _to_num(flows["pkts"]).clip(lower=0)
    denom = (pk.where(pk>=2, 2) - 1)  # avoid /0 and avoid exploding small pkts
    mean_iat = (dur / denom).replace([np.inf,-np.inf], np.nan).dropna()
    if mean_iat.empty:
        return None
    edges, cdf = _log10_hist_cdf(mean_iat, n_bins)
    if not edges: 
        return None
    return {"bins_log10": edges, "cdf": cdf}


# ---------- load and merge sources ----------
def load_all_sources(mawi_glob: str, unibs_glob: str, ugr16_file: str,
                     ugr16_days=None, ugr16_sample_frac=None) -> pd.DataFrame:
    frames = []
    # MAWI
    for path in glob.glob(mawi_glob):
        try:
            df_raw = read_zeek_conn_any(path)
            df = normalize_zeek_conn(df_raw)
            frames.append(df)
        except Exception as e:
            print(f"[WARN] MAWI {path}: {e}")
    # UNIBS
    for path in glob.glob(unibs_glob):
        try:
            df_raw = read_zeek_conn_any(path)
            df = normalize_zeek_conn(df_raw)
            frames.append(df)
        except Exception as e:
            print(f"[WARN] UNIBS {path}: {e}")
    # UGR'16
    if ugr16_file and os.path.exists(ugr16_file):
        try:
            df_ugr = read_ugr16_csv(ugr16_file, use_days=ugr16_days, sample_frac=ugr16_sample_frac)
            frames.append(df_ugr)
        except Exception as e:
            print(f"[WARN] UGR'16 {ugr16_file}: {e}")
    if not frames:
        raise FileNotFoundError("No input flows found. Check your globs and UGR'16 path.")
    flows = pd.concat(frames, ignore_index=True)
    return flows.replace([np.inf, -np.inf], np.nan)

flows = load_all_sources(MAWI_GLOB, UNIBS_GLOB, UGR16_FILE, UGR16_USE_DAYS, UGR16_SAMPLE_FRACTION)
print("Loaded flows:", len(flows))

def temporal_by_service(flows: pd.DataFrame,
                        top_k_services: int = 10,
                        min_rows_per_service: int = 200,
                        n_bins: int = 50):
    """
    Build per-service temporal references with index-safe masking.
    Returns {service: {"iat_flows": {...}, "flow_iat_mean": {...}, "count": int, "share": float}, ...}
    """
    if "service" not in flows.columns:
        return None

    # Unfiltered, index-aligned lowercase service column
    svc_all = flows["service"].astype(str).str.lower()
    # Define which entries are valid service labels for counting
    valid = ~svc_all.isin(["-", "none", "", "nan"])

    if not valid.any():
        return None

    # Use filtered copy ONLY for counts/top-K; keep svc_all for masks
    svc_counts = svc_all[valid].value_counts()
    if svc_counts.empty:
        return None

    top_svcs = list(svc_counts.head(top_k_services).index)
    total_valid = float(valid.sum())

    result = {}
    for svc in top_svcs:
        mask = (svc_all == svc)             # <-- index-aligned boolean mask
        if mask.sum() < min_rows_per_service:
            continue

        block = flows.loc[mask]             # <-- mask aligns with flows
        node = {
            "count": int(mask.sum()),
            "share": float(mask.sum() / total_valid)
        }

        # --- inter-flow IAT from start_ts
        if "start_ts" in block.columns:
            ts = pd.to_numeric(block["start_ts"], errors="coerce").dropna().sort_values()
            if len(ts) > 1:
                iat = ts.diff().dropna()
                iat = iat[iat > 0]
                if not iat.empty:
                    bins, cdf = _log10_hist_cdf(iat, n_bins)
                    if bins:
                        node["iat_flows"] = {"bins_log10": bins, "cdf": cdf}

        # --- within-flow IAT mean (explicit or proxy)
        vals = None
        if "flow_iat_mean_vals" in block.columns:
            vals = pd.to_numeric(block["flow_iat_mean_vals"], errors="coerce").dropna()
        if (vals is None or vals.empty) and {"duration", "pkts"}.issubset(block.columns):
            dur = pd.to_numeric(block["duration"], errors="coerce").clip(lower=0)
            pk  = pd.to_numeric(block["pkts"], errors="coerce").clip(lower=0)
            denom = (pk.where(pk >= 2, 2) - 1)
            vals = (dur / denom).replace([np.inf, -np.inf], np.nan).dropna()
        if vals is not None and not vals.empty:
            bins, cdf = _log10_hist_cdf(vals, n_bins)
            if bins:
                node["flow_iat_mean"] = {"bins_log10": bins, "cdf": cdf}

        if any(k in node for k in ("iat_flows", "flow_iat_mean")):
            result[svc] = node

    return result or None



# ---------- build baseline ----------
def build_baseline(flows: pd.DataFrame) -> Dict:
    coverage_hours = None
    if flows["start_ts"].notna().any():
        ts = _to_num(flows["start_ts"]).dropna()
        if not ts.empty:
            coverage_hours = float((ts.max() - ts.min()) / 3600.0)

    # metrics (include ONLY if not None / non-empty)
    volume = compute_volume(flows, n_bins=BINS_LOG10)
    iat    = compute_iat(flows, n_bins=BINS_LOG10)
    flow_iat_mean = compute_flow_iat_mean_proxy(flows, n_bins=BINS_LOG10)  # NEW
    burst  = compute_burstiness(flows, window_s=BURST_WINDOW_S)
    diurn  = compute_diurnal(flows)
    p_mix  = proto_mix(flows)
    s_mix  = service_mix(flows)

     # NEW: per-service temporal references
    svc_temporal = temporal_by_service(
        flows,
        top_k_services=10,
        min_rows_per_service=200,
        n_bins=BINS_LOG10
    )

    metrics = {}
    metrics.update(volume)
    if iat   is not None: metrics["iat_flows"] = iat
    if flow_iat_mean is not None:   metrics["flow_iat_mean"] = flow_iat_mean   # NEW
    if burst is not None: metrics["burstiness"] = burst
    if diurn is not None: metrics["diurnal_shape_hourly"] = diurn
    if p_mix:             metrics["protocol_mix"] = p_mix
    if s_mix is not None: metrics["service_mix"] = s_mix
    if svc_temporal is not None:         metrics["temporal_by_service"] = svc_temporal  # <-- NEW

    now = datetime.utcnow()
    baseline = {
        "profile": PROFILE_NAME,
        "created_at_utc": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "version": now.strftime("%Y-%m"),
        "sources": {
            "mawi_glob": MAWI_GLOB,
            "ugr16_file": UGR16_FILE,
            "unibs_glob": UNIBS_GLOB,
        },
        "stats": {
            "n_flows": int(len(flows)),
            "coverage_hours": coverage_hours,
        },
        "metrics": metrics,
        "notes": {
            "winsorize_quantiles": [0.005, 0.995],
            "bins": BINS_LOG10,
            "burst_window_s": BURST_WINDOW_S,
            "ugr16_sample_fraction": UGR16_SAMPLE_FRACTION,
            "ugr16_days_used": UGR16_USE_DAYS,
        },
    }
    return baseline

baseline = build_baseline(flows)
print("Metrics available:", list(baseline["metrics"].keys()))
print("n_flows:", baseline["stats"]["n_flows"])
print("coverage_hours:", baseline["stats"]["coverage_hours"])

# ---------- save YAML ----------
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, f"{PROFILE_NAME}_baseline_{datetime.utcnow().strftime('%Y%m%d')}.yaml")
with open(out_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(baseline, f, sort_keys=False, allow_unicode=True)
print("Baseline written to:", out_path)


C:\Users\ifrah\AppData\Local\Temp\ipykernel_25212\2008954995.py:198: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  te = pd.to_datetime(df["te"], errors="coerce", utc=True).view("int64")/1e9


Loaded flows: 22750949
Metrics available: ['bytes_per_flow', 'pkts_per_flow', 'flow_duration', 'iat_flows', 'flow_iat_mean', 'burstiness', 'diurnal_shape_hourly', 'protocol_mix', 'service_mix', 'temporal_by_service']
n_flows: 22750949
coverage_hours: 81149.07795952645
Baseline written to: baselines\general_enterprise_nids_baseline_20251016.yaml


In [14]:
# ==== Baseline sanity checker ====
import yaml, numpy as np, pandas as pd

YAML_PATH = out_path  # or set explicitly: r"baselines/general_enterprise_nids_baseline_YYYYMMDD.yaml"

with open(YAML_PATH, "r", encoding="utf-8") as f:
    bl = yaml.safe_load(f)

print("Profile:", bl.get("profile"))
print("Version:", bl.get("version"))
print("Flow count:", bl.get("stats",{}).get("n_flows"))
print("\nMetrics present:")
for k in bl.get("metrics",{}).keys():
    print(" •", k)

m = bl["metrics"]

def show_cdf_info(name, node):
    if node is None: 
        print(f"{name}: MISSING"); return
    if "bins_log10" in node:
        n = len(node["cdf"])
        print(f"{name}: log10 CDF with {n} bins")
    elif "bins" in node:
        n = len(node["cdf"])
        print(f"{name}: linear CDF with {n} bins")
    else:
        print(f"{name}: structure = {list(node.keys())}")

print("\n— Volume CDFs —")
show_cdf_info("bytes_per_flow", m.get("bytes_per_flow"))
show_cdf_info("pkts_per_flow",  m.get("pkts_per_flow"))
show_cdf_info("flow_duration",  m.get("flow_duration"))

print("\n— Temporal —")
show_cdf_info("iat_flows",      m.get("iat_flows"))
if "burstiness" in m:
    b = m["burstiness"]
    print(f"burstiness: Fano={b.get('fano_ref'):.4f}, bucket_s={b.get('bucket_s', bl.get('notes',{}).get('burst_window_s'))}")
else:
    print("burstiness: MISSING")
if "diurnal_shape_hourly" in m:
    dh = np.array(m["diurnal_shape_hourly"], dtype=float)
    print(f"diurnal_shape_hourly: length={len(dh)}, sum={dh.sum():.3f} (should be ~1.0)")
else:
    print("diurnal_shape_hourly: MISSING")

print("\n— Mixes —")
pm = m.get("protocol_mix")
sm = m.get("service_mix")
if pm: print("protocol_mix: keys:", sorted(pm.keys())[:10], "…", f"(total={len(pm)})")
else:  print("protocol_mix: MISSING or empty")
if sm: print("service_mix: keys:", sorted(sm.keys())[:10], "…", f"(total={len(sm)})")
else:  print("service_mix: MISSING or empty")


Profile: general_enterprise_nids
Version: 2025-10
Flow count: 22750949

Metrics present:
 • bytes_per_flow
 • pkts_per_flow
 • flow_duration
 • iat_flows
 • flow_iat_mean
 • burstiness
 • diurnal_shape_hourly
 • protocol_mix
 • service_mix
 • temporal_by_service

— Volume CDFs —
bytes_per_flow: log10 CDF with 50 bins
pkts_per_flow: log10 CDF with 50 bins
flow_duration: log10 CDF with 50 bins

— Temporal —
iat_flows: log10 CDF with 50 bins
burstiness: Fano=211847.5940, bucket_s=10
diurnal_shape_hourly: length=24, sum=1.000 (should be ~1.0)

— Mixes —
protocol_mix: keys: ['esp', 'gre', 'icmp', 'ipip', 'ipv6', 'tcp', 'udp'] … (total=7)
service_mix: keys: ['ayiya', 'dns', 'nan', 'vxlan'] … (total=4)
